In [1]:
import numpy as np
import pandas as pd
import os
import glob
import itertools
import scanpy as sc
import natsort
import json

import matplotlib.pyplot as plt
import seaborn as sns

from scroutines import basicu
from scroutines import powerplots

import scanpy.external as sce


In [2]:
ddir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_dev_merged'
outdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/astro_john/scenicplus_inputdata/'
!ls $ddir
!ls $outdir

bigtensor_cheng22_subclass_v2.json  bigtensor_yoo25_subclass_v2.npy
bigtensor_cheng22_subclass_v2.npy   cheng22_astro.h5ad
bigtensor_gao25_subclass.json	    gao25_astro.h5ad
bigtensor_gao25_subclass.npy	    yoo25_astro.h5ad
bigtensor_yoo25_subclass_v2.json
barcodes


In [3]:
adata = sc.read(os.path.join(ddir, 'yoo25_astro.h5ad'))
print(adata)

AnnData object with n_obs × n_vars = 14028 × 16572
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'cond', 'biosample'
    var: 'feature_types'


In [4]:
def streamline_barcode(x):
    return x.split(' ')[0][:-len("-2023")]


cell_barcodes = adata.obs_names.values
vfunc  = np.vectorize(streamline_barcode)
cell_barcodes_simp = vfunc(cell_barcodes)

assert len(np.unique(cell_barcodes_simp)) == len(cell_barcodes)
cell_barcodes_simp

array(['TCTTAGTTCAGCAAAG-1-P6b', 'GGAACAATCTTAGGGT-1-P6c',
       'TGGCCTTTCCGTTATT-1-P6a', ..., 'CGGACCTAGGCTTAAC-1-P21DRa',
       'GAGGGAGCATAATGAG-1-P21DRb', 'GAGAAACGTTAGCATG-1-P21DRa'],
      dtype='<U25')

In [5]:
adata.obs_names = cell_barcodes_simp

In [6]:
adata.obs

,Age,Doublet,Doublet Score,n_counts,n_genes,percent_mito,sample,Type,Subclass,Class,Sample,total_counts,pct_counts_mt,n_genes_by_counts,total_counts_mt,Doublet?,Study,Type_leiden,cond,biosample
TCTTAGTTCAGCAAAG-1-P6b,P6,False,0.007787,NaN,NaN,NaN,NaN,Astro_B,Astro,Non-neurons,P6b,1435.0,0.836237,865.0,12.0,0.0,2023 Multiome,Astro_A,P6NR,P6b
GGAACAATCTTAGGGT-1-P6c,P6,False,0.004915,NaN,NaN,NaN,NaN,Astro_A,Astro,Non-neurons,P6c,1274.0,1.648352,867.0,21.0,0.0,2023 Multiome,Astro_A,P6NR,P6c
TGGCCTTTCCGTTATT-1-P6a,P6,False,0.004764,NaN,NaN,NaN,NaN,Astro_A,Astro,Non-neurons,P6a,2165.0,3.048499,1320.0,66.0,0.0,2023 Multiome,Astro_A,P6NR,P6a
AAGGTATAGTTACCGG-1-P6c,P6,False,0.023792,NaN,NaN,NaN,NaN,Astro_Fem,Astro,Non-neurons,P6c,1729.0,6.824754,1163.0,118.0,0.0,2023 Multiome,Astro_Fem,P6NR,P6c
AGGGCTACAGTCTATG-1-P6c,P6,False,0.027423,NaN,NaN,NaN,NaN,Astro_Fem,Astro,Non-neurons,P6c,3305.0,1.331316,1918.0,44.0,0.0,2023 Multiome,Astro_Fem,P6NR,P6c
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GGTGAGCCACCAACCG-1-P21DRa,P21DR,False,0.011288,NaN,NaN,NaN,NaN,Astro_Fem2,Astro,Non-neurons,P21DRa,2554.0,4.581049,1518.0,117.0,0.0,2023 Multiome,Astro_Fem2,P21DR,P21DRa
GGTTGAGCACTAAGCC-1-P21DRb,P21DR,False,0.010065,NaN,NaN,NaN,NaN,Astro_Fem2,Astro,Non-neurons,P21DRb,3263.0,3.708244,1780.0,121.0,0.0,2023 Multiome,Astro_Fem2,P21DR,P21DRb
CGGACCTAGGCTTAAC-1-P21DRa,P21DR,False,0.004323,NaN,NaN,NaN,NaN,Astro_Fem1,Astro,Non-neurons,P21DRa,1647.0,1.578628,1009.0,26.0,0.0,2023 Multiome,Astro_Fem1,P21DR,P21DRa
GAGGGAGCATAATGAG-1-P21DRb,P21DR,False,0.007982,NaN,NaN,NaN,NaN,Astro_Fem2,Astro,Non-neurons,P21DRb,3465.0,8.542568,1899.0,296.0,0.0,2023 Multiome,Astro_Fem2,P21DR,P21DRb


In [7]:
adata.obs['Age'].unique()

['P6', 'P8', 'P10', 'P12', 'P12DR', ..., 'P14DR', 'P17', 'P17DR', 'P21', 'P21DR']
Length: 11
Categories (11, object): ['P6', 'P8', 'P10', 'P12', ..., 'P17', 'P17DR', 'P21', 'P21DR']

In [8]:
adata.X.data

array([2., 2., 5., ..., 2., 1., 3.], dtype=float32)

In [9]:
rna = adata

In [10]:
np.random.seed(0)

# Basic QC
sc.pp.filter_cells(rna, min_genes=200)
sc.pp.filter_genes(rna, min_cells=3)
rna.var["mt"] = rna.var_names.str.startswith("mt-")  # mouse uses lowercase "mt-"
sc.pp.calculate_qc_metrics(rna, qc_vars=["mt"], inplace=True)
rna = rna[rna.obs.pct_counts_mt < 20].copy()

# Normalise & cluster
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, n_top_genes=3000)
rna.raw = rna
rna = rna[:, rna.var.highly_variable].copy()
sc.pp.scale(rna, max_value=10)
sc.tl.pca(rna, n_comps=50)
sc.pp.neighbors(rna)
sc.tl.umap(rna)
sc.tl.leiden(rna, resolution=0.5)

rna.write_h5ad(os.path.join(outdir, "rna_preprocessed_astro_yoo25.h5ad"))
print(f"  RNA: {rna.shape[0]} cells × {rna.shape[1]} genes")


  RNA: 14026 cells × 3000 genes
